# Nodal energy price $\pi_k = \partial\mathcal{L}/\partial P_{d,k}$ — steady-state and dynamic components

In a conventional OPF the nodal price *is* the power-balance dual, because the balance is the
only constraint containing the load. **In a TSC-OPF that is false.** The load also enters the
transient block, so

$$\pi_k \;=\; \underbrace{\lambda_k}_{\text{steady state}} \;+\; \underbrace{\sum_{\text{dynamic rows } r} \Big(-\mu_r\,\frac{\partial \mathrm{expr}_r}{\partial P_{d,k}}\Big)}_{\text{dynamic / stability}}$$

This notebook reads **one finished run** and reports that split per bus. It solves nothing and
needs no extra files: every ingredient is already exported.

## Sign convention

JuMP's Lagrangian is $\mathcal{L} = f - \sum_r \mu_r(\mathrm{LHS}_r-\mathrm{RHS}_r)$ with every row
stored as $\mathrm{expr}\ \{=,\le\}\ 0$, so each row contributes $-\mu_r\,\partial\mathrm{expr}_r/\partial P_d$.
The steady-state balance is coded `Pg_sum - p_d - Σterms == 0`, hence
$\partial\mathrm{expr}/\partial P_d = -1$ and the energy term is $\;+\lambda_k\;$ — the exported
`dual_P_balance.csv` column **as is**.

`ABOUT ME/formulations.md` and the docs site said $\pi_k=-\lambda_k$ until 2026-08-16 and now
agree with the line above. The $|\text{energy}-\lambda|$ figure printed below is what actually
holds this down, so read that rather than trusting either document. Note also that the
**explicit dual LP** written to `Dispatch_Dual/` is a *different* object: it codes stationarity
as $c_g+\lambda_k=0$, so its multiplier is the negative of this one and there
$\pi_k=-\lambda_k$ is correct. Nothing in this notebook reads that file.

## Where the load enters the dynamics

| network form | dynamic rows containing $P_d$ | coefficient $\partial\mathrm{expr}/\partial P_{d,k}$ |
|---|---|---|
| `FULL_BUS` | nodal KCL `eq_const_Pbalance_tf/_tpf`, dual $\rho_k^t$ | $Z_k^t = a_Z V_{t,k}^2 + a_I V_{t,k}V_k + a_P V_k^2$ |
| `KRON_REDUCED` | `eq_const_Pe_tf/_tpf`, dual $\chi_g^t$ | $-E_g\sum_i E_i\big(\Re\,\mathrm{d}Y_{gi}\cos\Delta\delta + \Im\,\mathrm{d}Y_{gi}\sin\Delta\delta\big)$ |

with `zip_load_p` $=(a_Z,a_I,a_P)$ — **(Z, I, P) order**, so the default `(1,0,0)` is constant
impedance and the coefficient is $V_{t,k}^2$. On Kron the load becomes a shunt that the Schur
complement spreads over the whole reduced matrix, $\mathrm{d}Y_{\mathrm{red}}/\mathrm{d}y_k = u_k v_k^{\!\top}$.

Those two coefficients carry opposite signs, which is why the code below computes $-\rho Z$ on
FULL_BUS but $+\chi\,M$ on Kron: the transient KCL row is oriented the other way round from the
steady-state balance. Both match the column above; neither is a stray negation.

**Zero-load buses (FULL_BUS only).** Where the base load is zero the builder writes the row
*without* its $V_k^2$ factor, so its dual carries the opposite scaling and the coefficient is
$Z_k^t/V_k^2$. Handled below.

Both coefficient formulas were verified against a model-rebuild finite difference — $1.4\times10^{-14}$
on FULL_BUS, $5\times10^{-7}$ on Kron — and the resulting $\pi_k$ against a finite difference on the
objective ($1.7\times10^{-8}$ relative).

In [ ]:
import os, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

# --- locate the run -------------------------------------------------------------------
# KKT_RUN_DIR may name a run folder directly, or a RESULTS root (newest suitable run wins).
NEEDED = 'Transient_Stability/CSV_duals'
env = os.environ.get('KKT_RUN_DIR', '')
cand = pathlib.Path(env) if env else pathlib.Path.cwd()

def is_run(p):
    return (p / 'run_manifest.toml').is_file() and (p / NEEDED).is_dir()

if is_run(cand):
    RUN = cand
else:
    runs = sorted(p for p in cand.glob('Results - *') if is_run(p))
    if not runs:
        raise SystemExit(f'No transient run with dual exports found under {cand}. '
                         'Set KKT_RUN_DIR to a run folder or a RESULTS root.')
    RUN = runs[-1]

rd = lambda *p: pd.read_csv(RUN.joinpath(*p), sep=';')

# --- manifest: the run describes itself -------------------------------------------------
man = {}
for line in (RUN / 'run_manifest.toml').read_text(encoding='utf8').splitlines():
    s = line.strip()
    if '=' in s and not s.startswith('['):
        k, _, v = s.partition('=')
        man.setdefault(k.strip(), v.strip().strip('"'))

def zip_of(key, default=(1.0, 0.0, 0.0)):
    raw = man.get(key)
    if not raw:
        return default
    return tuple(float(x) for x in raw.strip('[]').split(','))

NETWORK  = man.get('network_form', 'KRON_REDUCED')
GENORDER = man.get('gen_order', 'CLASSICAL_2ND')
ZIP_P    = zip_of('zip_load_p')
LF       = float(man.get('load_factor', 1.0))   # scales the archived load table
N_TF     = int(float(man.get('n_steps_fault', 0)))
N_TPF    = int(float(man.get('n_steps_postfault', 0)))

print(f'run          : {RUN.name}')
print(f'case         : {man.get("case","?")}   load_factor {man.get("load_factor","?")}')
print(f'path         : {NETWORK} / {GENORDER}')
print(f'zip_load_p   : {ZIP_P}   (Z, I, P)')
print(f'time steps   : {N_TF} fault + {N_TPF} post-fault')
print(f'termination  : {man.get("termination_status","?")}')

In [ ]:
# --- steady-state side: identical on every path -----------------------------------------
buses = rd('Dispatch', 'CSV', 'buses_report.csv')
lam_df = rd('Dispatch', 'CSV_duals', 'dual_P_balance.csv')
lam = dict(zip(lam_df.iloc[:, 0].astype(int), lam_df.iloc[:, 1].astype(float)))

BUS_IDS = buses['BUS'].astype(int).tolist()
V_ss    = dict(zip(BUS_IDS, buses['V_pu'].astype(float)))
PD_MW   = dict(zip(BUS_IDS, buses['PD_MW'].astype(float)))

print(f'{len(BUS_IDS)} buses;  lambda range '
      f'{min(lam.values()):.4f} .. {max(lam.values()):.4f} EUR/MW')

In [ ]:
# --- dynamic side: coefficients from the shared module, contraction here ----------------
# `pd_coeffs.coeff_arrays` returns d(expr)/dP_d for every dynamic row, per perturbed bus.
# It is shared with the Tier 2-D notebook, which needs the same numbers as a MATRIX rather
# than contracted with duals — one implementation, so the two cannot drift apart.
#
# Both paths then apply the single MOI rule, identical to the generator side:
#
#       contribution = -dual x d(expr)/dP_d
#
import sys, pathlib

_here = pathlib.Path.cwd()
for _cand in (_here, _here / 'private' / 'kkt', pathlib.Path(r'..\..\private\kkt')):
    if (_cand / 'pd_coeffs.py').is_file():
        sys.path.insert(0, str(_cand.resolve()))
        break
else:
    raise SystemExit('pd_coeffs.py not found; run this notebook from kkt/')
import pd_coeffs

# A run carrying a per-load table (load_data.csv) split each bus's demand into loads with
# different ZIP shares, and the balances were built from the per-bus demand-weighted mean.
# That mean, not the config tuple, is the load model whose derivative the dual multiplies.
LOADS   = pd_coeffs.read_load_table(RUN, load_factor=LF) if NETWORK == 'FULL_BUS' else None
ZIP_BUS = pd_coeffs.zip_by_bus(LOADS, ZIP_P)
if LOADS is not None:
    print(f'load table: {len(LOADS)} loads on {LOADS.bus.nunique()} buses')

_, COEFF = pd_coeffs.coeff_arrays(RUN, network=NETWORK, zip_p=ZIP_BUS,
                                  n_tf=N_TF, n_tpf=N_TPF)

# Each branch fills dyn_tf[bus], dyn_tpf[bus] — the stability contribution split by window.
dyn_tf, dyn_tpf = {b: 0.0 for b in BUS_IDS}, {b: 0.0 for b in BUS_IDS}

if NETWORK == 'FULL_BUS':
    # Load at bus k appears only in bus k's own nodal balance, dual rho_k^t.
    rho = rd('Transient_Stability', 'CSV_duals', 'dual_Pbalance.csv')
    for b in BUS_IDS:
        col = f'Bus_{b}'
        if col not in rho.columns or b not in COEFF:
            continue
        contrib = -rho[col].to_numpy(float) * COEFF[b]
        dyn_tf[b] = float(contrib[:N_TF].sum())
        dyn_tpf[b] = float(contrib[N_TF:].sum())

elif NETWORK == 'KRON_REDUCED':
    # The Schur complement folds one bus's load shunt into the whole reduced matrix, so
    # that bus reaches EVERY generator's Pe row. COEFF[b] is (step, machine).
    chi = rd('Transient_Stability', 'CSV_duals', 'dual_Pe.csv')   # Gen_1..Gen_n per step
    chi_v = chi.to_numpy(float)
    for b in BUS_IDS:
        C = COEFF.get(b)
        if C is None:
            continue
        for lo, hi, store in ((0, N_TF, dyn_tf), (N_TF, N_TF + N_TPF, dyn_tpf)):
            tot = 0.0
            for t in range(lo, min(hi, len(C))):
                tot += float(-(chi_v[t] * C[t]).sum())
            store[b] = tot

else:
    raise SystemExit(f'network_form {NETWORK} not handled by this notebook.')

print(f'dynamic contributions computed for {NETWORK}')

In [ ]:
# --- the price table --------------------------------------------------------------------
tab = pd.DataFrame({
    'bus':        BUS_IDS,
    'PD_MW':      [PD_MW[b] for b in BUS_IDS],
    'energy':     [lam[b] for b in BUS_IDS],
    'stab_fault': [dyn_tf[b] for b in BUS_IDS],
    'stab_postf': [dyn_tpf[b] for b in BUS_IDS],
})
tab['stability'] = tab.stab_fault + tab.stab_postf
tab['pi']        = tab.energy + tab.stability
tab['stab_%']    = 100 * tab.stability / tab.pi

print(f'Nodal price decomposition  [EUR/MW]   —  {RUN.name}\n')
print(tab.to_string(index=False, float_format=lambda x: f'{x:10.4f}'))

spread_lam = tab.energy.max() - tab.energy.min()
spread_pi  = tab.pi.max() - tab.pi.min()
print(f'\nlocational spread:  lambda {spread_lam:.4f}   ->   pi {spread_pi:.4f} EUR/MW'
      f'   ({spread_pi/spread_lam:.2f}x)')

load_buses = tab[tab.PD_MW > 0]
if len(load_buses):
    pay_lam = (load_buses.PD_MW * load_buses.energy).sum()
    pay_pi  = (load_buses.PD_MW * load_buses.pi).sum()
    print(f'load payment at lambda {pay_lam:12.2f} EUR/h')
    print(f'load payment at pi     {pay_pi:12.2f} EUR/h'
          f'   ({100*(pay_pi-pay_lam)/pay_lam:+.2f} %)')

tab.to_csv(RUN / 'kkt_Pd_prices.csv', sep=';', index=False)
print(f'\nwritten: {RUN / "kkt_Pd_prices.csv"}')

In [ ]:
# --- per-load prices: one lambda, several stability charges ------------------------------
# Two loads on the same bus pay the SAME energy price and DIFFERENT stability prices: the
# dual belongs to the bus, the coefficient belongs to the load. That gap is the whole
# reason for splitting a bus in the first place.
if LOADS is None:
    print('no load_data.csv in this run — the per-bus table above is the whole story.')
else:
    rho = rd('Transient_Stability', 'CSV_duals', 'dual_Pbalance.csv')
    CL = pd_coeffs.load_coeff_arrays(RUN, loads=LOADS, n_tf=N_TF, n_tpf=N_TPF)

    rows = []
    for r in LOADS.itertuples(index=False):
        lid, b = int(r.id), int(r.bus)
        c, col = CL.get(lid), f'Bus_{int(r.bus)}'
        if c is None or col not in rho.columns:
            continue
        contrib = -rho[col].to_numpy(float) * c
        rows.append({
            'load':       lid,
            'bus':        b,
            'PD_MW':      float(r.p_d),
            'aZ':         float(r.aZ_p),
            'aI':         float(r.aI_p),
            'aP':         float(r.aP_p),
            'energy':     lam[b],
            'stab_fault': float(contrib[:N_TF].sum()),
            'stab_postf': float(contrib[N_TF:].sum()),
        })

    ltab = pd.DataFrame(rows)
    ltab['stability'] = ltab.stab_fault + ltab.stab_postf
    ltab['pi']        = ltab.energy + ltab.stability
    print(f'Per-load price decomposition  [EUR/MW]   —  {RUN.name}\n')
    print(ltab.to_string(index=False, float_format=lambda x: f'{x:10.4f}'))

    # The identity that ties this table to the per-bus one. The model only ever saw the
    # demand-weighted mean, so the MW-weighted per-load charges must reproduce the bus
    # charge exactly; a mismatch means the notebook and the builder disagree about the
    # load model, which is precisely the drift this assert exists to catch.
    for b, grp in ltab.groupby('bus'):
        lhs = float((grp.PD_MW * grp.stability).sum())
        rhs = float(PD_MW[b] * (dyn_tf[b] + dyn_tpf[b]))
        assert abs(lhs - rhs) <= 1e-6 * max(1.0, abs(rhs)), (b, lhs, rhs)
    print('\nper-load charges aggregate to the per-bus charge — exact identity holds')

    ltab.to_csv(RUN / 'kkt_Pd_prices_by_load.csv', sep=';', index=False)
    print(f'written: {RUN / "kkt_Pd_prices_by_load.csv"}')


In [ ]:
# --- figures ----------------------------------------------------------------------------
# Displayed only. nbconvert embeds them in the HTML report, so writing standalone .svg
# files alongside it just duplicates the same picture into the run folder — the two
# generator-side notebooks never did, and this now matches them.
fig, ax = plt.subplots(figsize=(10, 4.6))
x = np.arange(len(tab))
ax.bar(x - 0.2, tab.energy,    width=0.4, label=r'$\lambda_k$  (steady state)', color='#4878a8')
ax.bar(x + 0.2, tab.stability, width=0.4, label=r'stability',                   color='#c44e52')
ax.plot(x, tab.pi, 'ko-', ms=5, lw=1.2, label=r'$\pi_k$  total')
ax.axhline(0, color='gray', lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels([f'{b}\n{p:.0f} MW' if p > 0 else f'{b}\n—'
                    for b, p in zip(tab.bus, tab.PD_MW)])
ax.set_xlabel('bus'); ax.set_ylabel('EUR/MW')
ax.set_title(f'Nodal price decomposition — {NETWORK} / {GENORDER} — {RUN.name}')
ax.legend(); fig.tight_layout()
plt.show()

fig2, ax2 = plt.subplots(figsize=(10, 3.6))
ax2.bar(x - 0.2, tab.stab_fault, width=0.4, label='fault window',      color='#dd8452')
ax2.bar(x + 0.2, tab.stab_postf, width=0.4, label='post-fault window', color='#55a868')
ax2.axhline(0, color='gray', lw=0.6)
ax2.set_xticks(x); ax2.set_xticklabels(tab.bus)
ax2.set_xlabel('bus'); ax2.set_ylabel('EUR/MW')
ax2.set_title('Stability component by time window')
ax2.legend(); fig2.tight_layout()
plt.show()

## Reading the numbers

**`energy` is the incumbent LMP**, exactly the exported `dual_P_balance` column. Everything a
stability-blind market would charge.

**`stability` is signed, and both signs are physical.** A positive value means load at that bus
makes the swing worse and should pay more. A *negative* value means load there helps — under a
constant-impedance ZIP model a load near the fault behaves as a braking resistance in the
post-fault window, absorbing accelerating energy. Buses at or near the faulted bus routinely
come out negative.

**Fault vs post-fault.** A bus held at ~zero voltage during a bolted fault draws no power under
constant-impedance load, so its fault-window coefficient $V_{t}^2 \to 0$ and its entire stability
term is post-fault. That is physics, not a missing term.

**The locational spread** printed above is the headline: the stability component typically widens
the price difference between buses well beyond what $\lambda$ alone shows, i.e. a conventional LMP
understates *where* load should sit.

## Caveats worth respecting

- **The corridor must actually bind.** If the stability constraint is slack its duals are ~0 and
  every stability number here is numerical noise. Check `max|dual_delta_COI_*|` before believing
  a small non-zero value.
- **Dual quality is the weak link.** The energy term is exact to solver precision, but the
  stability term is a sum over transient duals. A run terminating `ALMOST_LOCALLY_SOLVED` (typical
  with `hessian_approximation = "limited-memory"`) carries visibly looser multipliers; plain Ipopt
  with an exact Hessian converges better *and* faster on these cases.
- **This is a marginal price at the current optimum.** It is the correct derivative by the
  envelope theorem, but it is local: at an active-set change the value function is kinked and
  $\pi_k$ is one element of the subdifferential.
- **Kron overstates the stability term** relative to FULL_BUS (measured: 33–53 % on 9-bus), and
  the *totals* can agree while both channels are individually wrong, because errors in $\lambda$
  and in the stability term partly cancel. Compare the split, never the bottom line alone.

**Per-load rows, when the case has a `load_data.csv`.** Splitting a bus into loads changes no
dual: `energy` is the same LMP for every load on the bus. What differs is `stability`, because
the coefficient the bus dual multiplies is that load's own ZIP polynomial. A constant-power
load holds its draw through the voltage dip and is charged for it; a constant-impedance load
beside it sheds demand exactly when the network needs it to, and is charged less — or paid.
The MW-weighted per-load charges add back up to the bus charge, which the cell asserts.
